In [6]:
# Cell 1 — Import Libraries
import os
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_community.document_loaders import PyPDFLoader
from transformers import pipeline

print("✅ All RAG libraries loaded!")

✅ All RAG libraries loaded!


In [7]:
# Cell 2 — Download and Load PDF
print("⏳ Downloading PDF paper...")

# Download Attention is All You Need paper
url = "https://arxiv.org/pdf/1706.03762"
response = requests.get(url)

# Save it
os.makedirs('../data/raw', exist_ok=True)
with open('../data/raw/attention_paper.pdf', 'wb') as f:
    f.write(response.content)

print("✅ PDF downloaded!")

# Load PDF
loader = PyPDFLoader('../data/raw/attention_paper.pdf')
pages = loader.load()

print(f"✅ PDF loaded!")
print(f"Total pages: {len(pages)}")
print(f"\nSample text from page 1:")
print(pages[0].page_content[:300])

⏳ Downloading PDF paper...
✅ PDF downloaded!
✅ PDF loaded!
Total pages: 15

Sample text from page 1:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


In [8]:
# Cell 3 — Split Text into Chunks
print("⏳ Splitting text into chunks...")

# Split text into smaller chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print(f"✅ Text split successfully!")
print(f"Total chunks: {len(chunks)}")
print(f"\nSample chunk:")
print(chunks[0].page_content[:300])
print(f"\nChunk metadata: {chunks[0].metadata}")

⏳ Splitting text into chunks...
✅ Text split successfully!
Total chunks: 93

Sample chunk:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par

Chunk metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/raw/attention_paper.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


In [9]:
# Cell 4 — Create Vector Database
print("⏳ Loading embedding model...")
print("⚠️  This takes 2-3 mins — downloading embedding model...")

# Load embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded!")
print("⏳ Creating vector database...")

# Create FAISS vector store
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save it locally
os.makedirs('../models/saved/faiss_index', exist_ok=True)
vectorstore.save_local('../models/saved/faiss_index')

print(f"✅ Vector database created and saved!")
print(f"Total vectors stored: {vectorstore.index.ntotal}")

⏳ Loading embedding model...
⚠️  This takes 2-3 mins — downloading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!
⏳ Creating vector database...
✅ Vector database created and saved!
Total vectors stored: 93


In [13]:
# Cell 5 — Build RAG QA System
print("⏳ Loading QA model...")
print("⚠️  This takes 2-3 mins — downloading FLAN-T5...")

# Load QA model
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

tokenizer_t5 = T5Tokenizer.from_pretrained("google/flan-t5-base")
model_t5 = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

print("✅ QA model loaded!")

# Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

def ask_question(question):
    # Step 1 — Retrieve relevant chunks
    relevant_docs = retriever.invoke(question)
    
    # Step 2 — Build context from chunks
    context = "\n".join([doc.page_content for doc in relevant_docs])
    
    # Step 3 — Generate answer
    prompt = f"Answer the question based on the context.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
    
    inputs = tokenizer_t5(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = model_t5.generate(**inputs, max_new_tokens=150)
    answer = tokenizer_t5.decode(outputs[0], skip_special_tokens=True)
    
    return answer, relevant_docs

print("✅ Function updated!")

⏳ Loading QA model...
⚠️  This takes 2-3 mins — downloading FLAN-T5...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ QA model loaded!
✅ Function updated!


In [14]:
# Cell 6 — Ask Questions to RAG System
questions = [
    "What is the transformer architecture?",
    "What is multi-head attention?",
    "What optimizer was used in the experiments?",
    "What are the main contributions of this paper?"
]

print("🤖 RAG Question Answering System")
print("="*60)

for question in questions:
    answer, docs = ask_question(question)
    print(f"\n❓ Question: {question}")
    print(f"💡 Answer: {answer}")
    print(f"📄 Sources used: {len(docs)} chunks")
    print("-"*60)

print("\n✅ RAG system working successfully!")

🤖 RAG Question Answering System

❓ Question: What is the transformer architecture?
💡 Answer: stacked self-attention and point-wise, fully connected layers
📄 Sources used: 3 chunks
------------------------------------------------------------

❓ Question: What is multi-head attention?
💡 Answer: allows the model to jointly attend to information from different representation subspaces at different positions
📄 Sources used: 3 chunks
------------------------------------------------------------

❓ Question: What optimizer was used in the experiments?
💡 Answer: Adam optimizer
📄 Sources used: 3 chunks
------------------------------------------------------------

❓ Question: What are the main contributions of this paper?
💡 Answer: Attention Is All You Need
📄 Sources used: 3 chunks
------------------------------------------------------------

✅ RAG system working successfully!


In [15]:
# Cell 7 — Save Results
import json

results = {
    'system': 'RAG (Retrieval Augmented Generation)',
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
    'llm': 'google/flan-t5-base',
    'vector_db': 'FAISS',
    'document': 'Attention is All You Need (2017)',
    'total_chunks': 93,
    'questions_answered': 4,
    'status': 'working'
}

os.makedirs('../models/saved', exist_ok=True)
with open('../models/saved/rag_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print("✅ RAG results saved!")
print("\n📊 System Summary:")
for key, value in results.items():
    print(f"  {key}: {value}")

✅ RAG results saved!

📊 System Summary:
  system: RAG (Retrieval Augmented Generation)
  embedding_model: sentence-transformers/all-MiniLM-L6-v2
  llm: google/flan-t5-base
  vector_db: FAISS
  document: Attention is All You Need (2017)
  total_chunks: 93
  questions_answered: 4
  status: working
